In [1]:
import torch
import numpy as np
import pandas as pd
import re
import os
import matplotlib.pyplot as plt
import torchvision
import tqdm
import glob
from PIL import Image
import seaborn as sns

import sys
sys.path.append('../../vision')

import datasets
import training_loop
import losses
import models

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(device)

cuda


# Labels dataset

In [12]:
views_dir = '../../data/semi-processed/supervised/growing_multiview_training/'
views = sorted([os.path.basename(view) for view in glob.glob(views_dir + '*')])

In [13]:
views

['deep_tg_col_a',
 'dta_sax',
 'dta_sax_arch',
 'me_2_ch',
 'me_2_ch_col_m',
 'me_4_ch',
 'me_4_ch_col_m',
 'me_4_ch_col_t',
 'me_5_ch',
 'me_5_ch_col',
 'me_av_sax',
 'me_av_sax_col',
 'me_bicaval',
 'me_bicaval_col',
 'me_in_out',
 'me_in_out_col_p',
 'me_in_out_col_t',
 'me_lax',
 'me_lax_col_a',
 'me_lax_col_m',
 'me_lax_col_mv_av',
 'me_mit_com',
 'me_mit_com_col_m',
 'other',
 'tg_basal_sax',
 'tg_mid_pap_sax',
 'ue_asc_ao_sax']

In [ ]:
quality_label_df = pd.read_csv('../../labels/quality_grading_300.csv')

In [ ]:
class QualityDataset(torch.utils.data.Dataset):

    def __init__(self, labels_df, transform=None, duration=32, skip=1):
        self.labels_df = labels_df
        self.transform = transform
        self.duration = duration
        self.skip = skip

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):                  
        label = torch.tensor((self.labels_df.iloc[idx]['LV'] == 'y' , self.labels_df.iloc[idx]['RV'] == 'y'))
        
        data = self.__read_images(self.labels_df.iloc[idx]['clip_path'], self.duration, self.skip)
        
        if self.transform:
            data = self.transform(data)
            
        data = data.permute(1,0,2,3) # reshape to (C,T,H,W)
            
        return {'data':data, 'label':label}
    
    
    def __read_images(self, path, duration=32, skip=1):
        frame_paths = glob.glob(path+'/*')
        if len(frame_paths) <= duration * skip:
            start = 0
            stop = len(frame_paths)
            pad = (duration * skip - stop) // skip
        else:
            start = np.random.randint(0, len(frame_paths) - duration * skip + 1)
            stop = start + duration * skip
            pad = 0
        frames = []
        for frame in sorted(frame_paths)[start:stop:skip]:
            frames.append(torchvision.transforms.functional.to_tensor((Image.open(frame))))
        for i in range(pad):
            frames.append(torch.zeros_like(frames[-1]))
        frames = torch.stack(frames, dim=0)
        return frames
    
# plt.imshow(train_dataset[0]['data'][:,0].permute(1,2,0))